# TensorFly: real MaleCNS v1.0 × Qwen3.5-9B

This notebook runs a real, checksum-verified experiment. It downloads official MaleCNS tables and selected official skeletons automatically; it never uses a synthetic graph unless `synthetic_dev=True` is explicitly requested in a developer-only cell. Qwen runs here in Colab, not during local code inspection.

> TensorFly uses reconstructed MaleCNS v1.0 anatomy, neuron identities, morphology, and synaptic connectivity. Neural dynamics, metric encoding, reward modulation, plasticity and runtime actions are engineered approximations.


In [ ]:
# Idempotent setup: safe for Colab Run all and after a runtime reconnect.
from pathlib import Path
import os, subprocess, sys

repo = Path('/content/fly-inference-optimizer')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not the TensorFly git clone; remove it or choose a clean Colab runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/MrFaruk0/fly-inference-optimizer.git', str(repo)], check=True)
os.chdir(repo)

# --no-deps ensures the editable src/ package is installed even if an optional
# accelerator package has a resolver conflict. The second command installs the
# runtime dependencies into this exact notebook kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pyarrow', 'accelerate', 'safetensors', 'transformers'], check=True)

import tensorfly
print('TensorFly imported from:', tensorfly.__file__)
print('TensorFly version:', tensorfly.__version__)


In [ ]:
import tensorfly

# First run downloads ~1.1 GB of locked official tables and a deterministic
# real-skeleton browser subset. Later runs reuse hashes and derived arrays.
prepared = tensorfly.prepare()
print(prepared.dataset.report)
print(prepared.populations.to_dict())
print('real morphology:', prepared.viewer_morphology)


In [ ]:
from tensorfly import DEFAULT_PROMPTS, TensorFlyExperiment

# Model weights load lazily only when run() invokes the actual benchmark.
experiment = TensorFlyExperiment(model='Qwen/Qwen3.5-9B')
records = experiment.run(prompt_corpus=DEFAULT_PROMPTS, trials=20)
records[-1]


In [ ]:
# Equal model, prompts, warmup, evaluation count, and fixed output budget.
baselines = experiment.compare_baselines(prompt_corpus=DEFAULT_PROMPTS, trials=20)
{name: rows[-1].reward for name, rows in baselines.items()}


In [ ]:
# Replay is made only from recorded Qwen metrics, config changes and simulation states.
frames = experiment.replay()
replay_path = experiment.export_video()  # viewer/tensorfly_replay.json for browser capture
print(replay_path, len(frames))
!python -m http.server 8000 --directory viewer > /tmp/tensorfly-viewer.log 2>&1 &
